# Speech Adversarial Attack on an ASR Model

This notebook demonstrates an empirical adversarial attack against an automatic speech recognition model. The goal is to add a small waveform perturbation that keeps the audio perceptually similar to a human listener, while causing the ASR model to produce a worse transcription.

This is not certification. It is an attack/demo notebook, complementary to the DeepPoly MNIST certification notebook.

The notebook has two parts: first, a gradient-based waveform attack; second, a safe SACRED-Bench-inspired composition attack section that uses benign overlapping speech and benign background audio.

**Model choice.** We use `facebook/wav2vec2-base-960h`, a free Apache-2.0 Hugging Face CTC speech-recognition model. CTC models expose frame-level logits, which makes a gradient-based waveform attack simpler and more transparent than attacking a sequence-to-sequence model such as Whisper.

**Attack objective.** The attack is untargeted: by default it maximizes CTC loss against the model's clean transcript, while constraining the waveform perturbation inside a small `L_inf` radius. Using the clean transcript as the attack reference makes the demo measure output instability directly, even when the ASR model is already imperfect on the human reference.


## Colab Setup

Run the next cell first if using Google Colab. After installation, continue with the rest of the notebook.


In [ ]:
%pip install transformers soundfile jiwer matplotlib librosa


## 1. Setup

If dependencies are missing, run this in a notebook cell or terminal first:

```python
%pip install transformers soundfile jiwer matplotlib librosa
```

The first full run will download the ASR model and one small audio file from Hugging Face.


In [ ]:
import importlib.util

required_packages = [
    "torch",
    "transformers",
    "soundfile",
    "jiwer",
    "IPython",
    "matplotlib",
    "numpy",
    "pandas",
]
missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing:
    raise ImportError(
        "Missing packages: " + ", ".join(missing) +
        "\nInstall with: %pip install transformers soundfile jiwer matplotlib librosa"
    )


In [ ]:
import math
import re
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from jiwer import wer
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "facebook/wav2vec2-base-960h"
SAMPLE_RATE = 16_000

AUDIO_URL = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
AUDIO_PATH = Path("mlk_sample.flac")
REFERENCE_TEXT = "I have a dream that one day this nation will rise up and live out the true meaning of its creed."
USE_CLEAN_TRANSCRIPT_AS_ATTACK_REFERENCE = True

ATTACK_EPSILON = 0.002
ATTACK_STEPS = 20
ATTACK_STEP_SIZE = ATTACK_EPSILON / 4

RUN_SWEEP = True
SWEEP_EPSILONS = [0.0005, 0.001, 0.002, 0.003]
SWEEP_STEPS = 8

print(f"device={DEVICE}")
print(f"model={MODEL_ID}")


## 2. Load Model and Speech Sample

The clean sample is downloaded directly from a small Hugging Face audio sample file, instead of using `load_dataset`. This avoids old dataset-script compatibility errors in newer `datasets` versions.

The attack is only meaningful when the model is reasonably correct on the clean audio.


In [ ]:
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

if not AUDIO_PATH.exists():
    urllib.request.urlretrieve(AUDIO_URL, AUDIO_PATH)

clean_audio, original_sample_rate = sf.read(AUDIO_PATH, dtype="float32")
if clean_audio.ndim > 1:
    clean_audio = clean_audio.mean(axis=1)

if original_sample_rate != SAMPLE_RATE:
    try:
        import librosa
    except ImportError as exc:
        raise ImportError(
            f"Audio sample rate is {original_sample_rate}, but the model expects {SAMPLE_RATE}. "
            "Install librosa for resampling: %pip install librosa"
        ) from exc
    clean_audio = librosa.resample(clean_audio, orig_sr=original_sample_rate, target_sr=SAMPLE_RATE).astype(np.float32)

human_reference_text = REFERENCE_TEXT
clean_audio = np.clip(clean_audio.astype(np.float32), -1.0, 1.0)
clean_waveform = torch.tensor(clean_audio, dtype=torch.float32, device=DEVICE).unsqueeze(0)

print(f"audio_url={AUDIO_URL}")
print(f"duration={len(clean_audio) / SAMPLE_RATE:.2f}s")
print(f"human_reference={human_reference_text}")
display(Audio(clean_audio, rate=SAMPLE_RATE))


In [ ]:
def normalize_text(text):
    text = text.upper()
    text = re.sub(r"[^A-Z' ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


@torch.no_grad()
def transcribe_waveform(waveform):
    if torch.is_tensor(waveform):
        array = waveform.detach().squeeze(0).cpu().numpy()
    else:
        array = np.asarray(waveform, dtype=np.float32)

    inputs = processor(array, sampling_rate=SAMPLE_RATE, return_tensors="pt", padding=True)
    inputs = {name: value.to(DEVICE) for name, value in inputs.items()}
    logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(predicted_ids)[0]


def transcript_wer(reference, prediction):
    ref = normalize_text(reference)
    pred = normalize_text(prediction)
    if not ref:
        return np.nan
    return wer(ref, pred)


clean_transcript = transcribe_waveform(clean_waveform)
if USE_CLEAN_TRANSCRIPT_AS_ATTACK_REFERENCE:
    attack_reference_text = clean_transcript
    attack_reference_mode = "model_clean_transcript"
else:
    attack_reference_text = human_reference_text
    attack_reference_mode = "human_reference"

clean_wer = transcript_wer(attack_reference_text, clean_transcript)
clean_wer_vs_human = transcript_wer(human_reference_text, clean_transcript)

pd.DataFrame([{
    "human_reference": normalize_text(human_reference_text),
    "clean_transcript": normalize_text(clean_transcript),
    "clean_wer_vs_human": clean_wer_vs_human,
    "attack_reference_mode": attack_reference_mode,
    "attack_reference": normalize_text(attack_reference_text),
    "clean_wer_vs_attack_reference": clean_wer,
}])


## 3. Untargeted PGD Attack

The attack updates the waveform in the direction that increases CTC loss for the correct transcript. After each step, the perturbation is projected back into an `L_inf` ball around the clean audio.

A successful attack should increase ASR error while keeping the perturbation small. We measure smallness using max perturbation and signal-to-noise ratio (SNR), then listen to the clean and adversarial audio.


In [ ]:
def differentiable_wav2vec2_normalize(waveform):
    mean = waveform.mean(dim=1, keepdim=True)
    var = waveform.var(dim=1, keepdim=True, unbiased=False)
    return (waveform - mean) / torch.sqrt(var + 1e-7)


def encode_ctc_labels(text):
    normalized = normalize_text(text)
    labels = processor.tokenizer(normalized, return_tensors="pt").input_ids.to(DEVICE)
    return labels


def ctc_reference_loss(waveform, labels):
    input_values = differentiable_wav2vec2_normalize(waveform)
    logits = model(input_values).logits
    log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)

    batch_size = logits.shape[0]
    input_lengths = torch.full(
        size=(batch_size,),
        fill_value=logits.shape[1],
        dtype=torch.long,
        device=DEVICE,
    )
    target_lengths = torch.tensor([labels.shape[1]], dtype=torch.long, device=DEVICE)

    return F.ctc_loss(
        log_probs,
        labels,
        input_lengths,
        target_lengths,
        blank=processor.tokenizer.pad_token_id,
        reduction="mean",
        zero_infinity=True,
    )


def untargeted_pgd_attack(clean, reference, epsilon, steps, step_size):
    clean = clean.detach()
    adv = clean.clone().detach()
    labels = encode_ctc_labels(reference)
    history = []

    for step in range(1, steps + 1):
        adv = adv.detach().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        loss = ctc_reference_loss(adv, labels)
        grad = torch.autograd.grad(loss, adv)[0]

        adv = adv + step_size * grad.sign()
        perturbation = torch.clamp(adv - clean, min=-epsilon, max=epsilon)
        adv = torch.clamp(clean + perturbation, min=-1.0, max=1.0).detach()

        if step == 1 or step % 5 == 0 or step == steps:
            history.append({
                "step": step,
                "ctc_loss": float(loss.detach().cpu()),
                "max_abs_perturbation": float(perturbation.abs().max().detach().cpu()),
            })

    return adv, pd.DataFrame(history)


In [ ]:
adv_waveform, attack_history = untargeted_pgd_attack(
    clean_waveform,
    attack_reference_text,
    epsilon=ATTACK_EPSILON,
    steps=ATTACK_STEPS,
    step_size=ATTACK_STEP_SIZE,
)

attack_history


## 4. Attack Result

The main comparison is clean transcript vs adversarial transcript. `clean_wer` and `adversarial_wer` are measured against the configured attack reference. The perturbation is also played separately and amplified, so it is easier to inspect what was added.


In [ ]:
def snr_db(clean, adv):
    clean_np = clean.detach().squeeze(0).cpu().numpy()
    adv_np = adv.detach().squeeze(0).cpu().numpy()
    noise = adv_np - clean_np
    clean_rms = np.sqrt(np.mean(clean_np ** 2))
    noise_rms = np.sqrt(np.mean(noise ** 2))
    return 20 * np.log10((clean_rms + 1e-12) / (noise_rms + 1e-12))


def evaluate_attack(clean, adv, attack_reference, human_reference):
    clean_pred = transcribe_waveform(clean)
    adv_pred = transcribe_waveform(adv)
    clean_wer_value = transcript_wer(attack_reference, clean_pred)
    adv_wer_value = transcript_wer(attack_reference, adv_pred)
    noise = (adv - clean).detach().squeeze(0).cpu().numpy()

    return pd.DataFrame([{
        "epsilon": ATTACK_EPSILON,
        "attack_reference_mode": attack_reference_mode,
        "attack_reference": normalize_text(attack_reference),
        "clean_transcript": normalize_text(clean_pred),
        "adversarial_transcript": normalize_text(adv_pred),
        "clean_wer": clean_wer_value,
        "adversarial_wer": adv_wer_value,
        "wer_increase": adv_wer_value - clean_wer_value,
        "human_reference_wer_after_attack": transcript_wer(human_reference, adv_pred),
        "max_abs_perturbation": float(np.max(np.abs(noise))),
        "snr_db": snr_db(clean, adv),
        "attack_success": adv_wer_value > clean_wer_value,
    }])


attack_result = evaluate_attack(clean_waveform, adv_waveform, attack_reference_text, human_reference_text)
display(attack_result.style.format({
    "epsilon": "{:.4f}",
    "clean_wer": "{:.3f}",
    "adversarial_wer": "{:.3f}",
    "wer_increase": "{:+.3f}",
    "human_reference_wer_after_attack": "{:.3f}",
    "max_abs_perturbation": "{:.6f}",
    "snr_db": "{:.2f}",
}))


In [ ]:
adv_audio = adv_waveform.detach().squeeze(0).cpu().numpy()
noise = adv_audio - clean_audio

print("Clean audio")
display(Audio(clean_audio, rate=SAMPLE_RATE))

print("Adversarial audio")
display(Audio(adv_audio, rate=SAMPLE_RATE))

print("Perturbation only, amplified 20x for audibility")
display(Audio(np.clip(20 * noise, -1.0, 1.0), rate=SAMPLE_RATE))


In [ ]:
time = np.arange(len(clean_audio)) / SAMPLE_RATE
window = min(len(clean_audio), 2 * SAMPLE_RATE)

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=False)
axes[0].plot(time[:window], clean_audio[:window], linewidth=0.8)
axes[0].set_title("Clean waveform, first 2 seconds", loc="left")
axes[0].set_ylabel("amplitude")

axes[1].plot(time[:window], adv_audio[:window], linewidth=0.8, color="tab:orange")
axes[1].set_title("Adversarial waveform, first 2 seconds", loc="left")
axes[1].set_ylabel("amplitude")

axes[2].plot(time[:window], noise[:window], linewidth=0.8, color="tab:red")
axes[2].set_title("Perturbation, first 2 seconds", loc="left")
axes[2].set_xlabel("time (s)")
axes[2].set_ylabel("amplitude")

for ax in axes:
    ax.grid(alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()


## 5. Epsilon Sweep

The sweep checks how attack strength changes as the perturbation budget grows. WER is measured against the configured attack reference, and human-reference WER is shown separately. Higher epsilon usually gives a stronger attack but also makes the perturbation easier to hear.


In [ ]:
if RUN_SWEEP:
    rows = []
    for epsilon in SWEEP_EPSILONS:
        adv_eps, _ = untargeted_pgd_attack(
            clean_waveform,
            attack_reference_text,
            epsilon=epsilon,
            steps=SWEEP_STEPS,
            step_size=epsilon / 4,
        )
        pred = transcribe_waveform(adv_eps)
        adv_wer = transcript_wer(attack_reference_text, pred)
        noise_eps = (adv_eps - clean_waveform).detach().squeeze(0).cpu().numpy()
        rows.append({
            "epsilon": epsilon,
            "adversarial_transcript": normalize_text(pred),
            "adversarial_wer": adv_wer,
            "wer_increase": adv_wer - clean_wer,
            "human_reference_wer": transcript_wer(human_reference_text, pred),
            "max_abs_perturbation": float(np.max(np.abs(noise_eps))),
            "snr_db": snr_db(clean_waveform, adv_eps),
            "attack_success": adv_wer > clean_wer,
        })

    sweep = pd.DataFrame(rows)
    display(sweep.style.format({
        "epsilon": "{:.4f}",
        "adversarial_wer": "{:.3f}",
        "wer_increase": "{:+.3f}",
        "human_reference_wer": "{:.3f}",
        "max_abs_perturbation": "{:.6f}",
        "snr_db": "{:.2f}",
    }))
else:
    print("Set RUN_SWEEP=True to run the epsilon sweep.")


## 6. SACRED-Inspired Composition Attacks, Safe Version

SACRED-Bench studies black-box audio attacks based on realistic composition rather than optimized imperceptible noise. The original benchmark focuses on multimodal LLM safety; here we adapt the same *audio composition idea* to a safe ASR setting.

We do not use harmful content. Instead, we test whether benign composition changes the ASR transcript:

- **SSO-style:** clean carrier speech overlapped with another benign speech sample.
- **SAO-style:** clean carrier speech mixed with benign non-speech background audio.
- **MSD-style:** a simple two-speaker dialogue made from two benign speech clips.

For ASR, the relevant metric is not harmful compliance. We report whether the composed audio changes the transcript, increases WER against the clean transcript, or causes words from the second speech stream to appear.


In [ ]:
PAYLOAD_AUDIO_URL = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"
PAYLOAD_AUDIO_PATH = Path("payload_1.flac")


def load_audio_from_url(url, path, target_sample_rate=SAMPLE_RATE):
    if not path.exists():
        urllib.request.urlretrieve(url, path)

    audio, sample_rate = sf.read(path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if sample_rate != target_sample_rate:
        try:
            import librosa
        except ImportError as exc:
            raise ImportError(
                f"Audio sample rate is {sample_rate}, but the target is {target_sample_rate}. "
                "Install librosa for resampling: %pip install librosa"
            ) from exc
        audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=target_sample_rate).astype(np.float32)

    return np.clip(audio.astype(np.float32), -1.0, 1.0)


payload_audio = load_audio_from_url(PAYLOAD_AUDIO_URL, PAYLOAD_AUDIO_PATH)
payload_waveform = torch.tensor(payload_audio, dtype=torch.float32, device=DEVICE).unsqueeze(0)
payload_transcript = transcribe_waveform(payload_waveform)

print(f"payload_audio_url={PAYLOAD_AUDIO_URL}")
print(f"payload_duration={len(payload_audio) / SAMPLE_RATE:.2f}s")
print(f"payload_transcript={normalize_text(payload_transcript)}")
display(Audio(payload_audio, rate=SAMPLE_RATE))


In [ ]:
STOPWORDS = {
    "A", "AN", "AND", "ARE", "AS", "AT", "BE", "BUT", "BY", "FOR", "FROM", "HAVE",
    "HE", "I", "IN", "IS", "IT", "ITS", "ME", "MY", "OF", "ON", "OR", "OUR", "THE",
    "THIS", "TO", "UP", "WE", "WILL", "WITH", "YOU", "YOUR",
}


def db_to_gain(db):
    return 10 ** (db / 20)


def rms(audio):
    return float(np.sqrt(np.mean(np.square(audio)) + 1e-12))


def peak_normalize(audio, peak=0.95):
    max_abs = float(np.max(np.abs(audio)))
    if max_abs <= peak or max_abs < 1e-12:
        return audio.astype(np.float32)
    return (audio * (peak / max_abs)).astype(np.float32)


def fit_audio_length(audio, target_len):
    if len(audio) >= target_len:
        return audio[:target_len].astype(np.float32)
    repeats = math.ceil(target_len / len(audio))
    return np.tile(audio, repeats)[:target_len].astype(np.float32)


def time_stretch_audio(audio, speed):
    if abs(speed - 1.0) < 1e-6:
        return audio.astype(np.float32)
    try:
        import librosa
        stretched = librosa.effects.time_stretch(audio.astype(np.float32), rate=speed)
        return stretched.astype(np.float32)
    except Exception:
        old_x = np.linspace(0, 1, num=len(audio), endpoint=False)
        new_len = max(1, int(len(audio) / speed))
        new_x = np.linspace(0, 1, num=new_len, endpoint=False)
        return np.interp(new_x, old_x, audio).astype(np.float32)


def fade_envelope(length, fade_ms):
    envelope = np.ones(length, dtype=np.float32)
    fade_len = min(int(SAMPLE_RATE * fade_ms / 1000), length // 2)
    if fade_len > 1:
        envelope[:fade_len] = np.linspace(0.0, 1.0, fade_len, dtype=np.float32)
        envelope[-fade_len:] = np.linspace(1.0, 0.0, fade_len, dtype=np.float32)
    return envelope


def overlay_speech(carrier, payload, start_sec=1.0, payload_db=-8, speed=1.0, fade_ms=300):
    output = carrier.copy().astype(np.float32)
    start = int(start_sec * SAMPLE_RATE)
    if start >= len(output):
        raise ValueError("start_sec is beyond the carrier audio length")

    payload_mod = time_stretch_audio(payload, speed)
    payload_mod = payload_mod[: len(output) - start]
    if len(payload_mod) == 0:
        raise ValueError("payload audio is empty after trimming")

    carrier_segment = output[start:start + len(payload_mod)]
    relative_gain = (rms(carrier_segment) / rms(payload_mod)) * db_to_gain(payload_db)
    payload_layer = payload_mod * relative_gain * fade_envelope(len(payload_mod), fade_ms)

    output[start:start + len(payload_layer)] += payload_layer
    return peak_normalize(output)


def make_benign_background(length, kind="beeps"):
    t = np.arange(length, dtype=np.float32) / SAMPLE_RATE
    if kind == "beeps":
        background = np.zeros(length, dtype=np.float32)
        for start_sec in np.arange(0.4, len(t) / SAMPLE_RATE, 0.8):
            start = int(start_sec * SAMPLE_RATE)
            stop = min(length, start + int(0.12 * SAMPLE_RATE))
            if stop > start:
                burst_t = np.arange(stop - start, dtype=np.float32) / SAMPLE_RATE
                background[start:stop] += 0.5 * np.sin(2 * np.pi * 880 * burst_t)
        return background
    if kind == "tone":
        return (0.35 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
    rng = np.random.default_rng(SEED)
    return rng.normal(0.0, 0.25, size=length).astype(np.float32)


def mix_background(carrier, background, background_db=-15):
    background = fit_audio_length(background, len(carrier))
    gain = (rms(carrier) / rms(background)) * db_to_gain(background_db)
    return peak_normalize(carrier + background * gain)


def content_words(text):
    words = normalize_text(text).split()
    return {word for word in words if len(word) > 2 and word not in STOPWORDS}


def payload_keyword_metrics(payload_text, clean_text, composed_text):
    payload = content_words(payload_text)
    clean = content_words(clean_text)
    composed = content_words(composed_text)
    candidate_words = sorted(payload - clean)
    hit_words = [word for word in candidate_words if word in composed]
    hit_rate = len(hit_words) / len(candidate_words) if candidate_words else 0.0
    return hit_words, hit_rate


def evaluate_composed_audio(audio, attack_type, payload_text="", **metadata):
    waveform = torch.tensor(audio, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    transcript = transcribe_waveform(waveform)
    wer_vs_clean = transcript_wer(clean_transcript, transcript)
    hit_words, hit_rate = payload_keyword_metrics(payload_text, clean_transcript, transcript) if payload_text else ([], 0.0)

    row = {
        "attack_type": attack_type,
        **metadata,
        "transcript": normalize_text(transcript),
        "wer_vs_clean_transcript": wer_vs_clean,
        "payload_keyword_hit_rate": hit_rate,
        "payload_words_detected": " ".join(hit_words),
        "snr_db": snr_db(clean_waveform, torch.tensor(audio, dtype=torch.float32, device=DEVICE).unsqueeze(0)),
        "transcript_changed": wer_vs_clean > 0,
    }
    return row


In [ ]:
SSO_CONFIGS = [
    {"fade_ms": 300, "payload_speed": 1.3, "payload_db": -8},
    {"fade_ms": 500, "payload_speed": 1.3, "payload_db": -8},
    {"fade_ms": 700, "payload_speed": 1.3, "payload_db": -8},
    {"fade_ms": 500, "payload_speed": 1.1, "payload_db": -8},
    {"fade_ms": 500, "payload_speed": 1.5, "payload_db": -8},
    {"fade_ms": 500, "payload_speed": 1.3, "payload_db": -6},
    {"fade_ms": 500, "payload_speed": 1.3, "payload_db": -10},
]

sso_rows = []
sso_examples = []
for config in SSO_CONFIGS:
    composed = overlay_speech(
        clean_audio,
        payload_audio,
        start_sec=1.0,
        payload_db=config["payload_db"],
        speed=config["payload_speed"],
        fade_ms=config["fade_ms"],
    )
    sso_examples.append((config, composed))
    sso_rows.append(evaluate_composed_audio(
        composed,
        attack_type="SSO_safe",
        payload_text=payload_transcript,
        fade_ms=config["fade_ms"],
        payload_speed=config["payload_speed"],
        payload_db=config["payload_db"],
    ))

sso_results = pd.DataFrame(sso_rows)
display(sso_results.style.format({
    "payload_speed": "{:.1f}",
    "payload_db": "{:.0f}",
    "wer_vs_clean_transcript": "{:.3f}",
    "payload_keyword_hit_rate": "{:.2%}",
    "snr_db": "{:.2f}",
}))


In [ ]:
best_sso_idx = int(sso_results["wer_vs_clean_transcript"].idxmax())
best_sso_config, best_sso_audio = sso_examples[best_sso_idx]

print("Clean carrier audio")
display(Audio(clean_audio, rate=SAMPLE_RATE))

print("Payload speech audio")
display(Audio(payload_audio, rate=SAMPLE_RATE))

print(f"Best SSO-style composed audio: {best_sso_config}")
display(Audio(best_sso_audio, rate=SAMPLE_RATE))


In [ ]:
SAO_CONFIGS = [
    {"background_kind": "beeps", "background_db": -20},
    {"background_kind": "beeps", "background_db": -15},
    {"background_kind": "beeps", "background_db": -10},
    {"background_kind": "tone", "background_db": -15},
    {"background_kind": "noise", "background_db": -20},
]

sao_rows = []
sao_examples = []
for config in SAO_CONFIGS:
    background = make_benign_background(len(clean_audio), kind=config["background_kind"])
    composed = mix_background(clean_audio, background, background_db=config["background_db"])
    sao_examples.append((config, composed, background))
    sao_rows.append(evaluate_composed_audio(
        composed,
        attack_type="SAO_safe",
        background_kind=config["background_kind"],
        background_db=config["background_db"],
    ))

sao_results = pd.DataFrame(sao_rows)
display(sao_results.style.format({
    "background_db": "{:.0f}",
    "wer_vs_clean_transcript": "{:.3f}",
    "payload_keyword_hit_rate": "{:.2%}",
    "snr_db": "{:.2f}",
}))


In [ ]:
best_sao_idx = int(sao_results["wer_vs_clean_transcript"].idxmax())
best_sao_config, best_sao_audio, best_background = sao_examples[best_sao_idx]

print(f"Best SAO-style composed audio: {best_sao_config}")
display(Audio(best_sao_audio, rate=SAMPLE_RATE))

print("Background layer only, amplified 4x for inspection")
display(Audio(np.clip(4 * fit_audio_length(best_background, len(clean_audio)), -1.0, 1.0), rate=SAMPLE_RATE))


In [ ]:
def build_safe_dialogue_audio(first_speaker, second_speaker, segment_sec=5.0, pause_sec=0.25):
    segment_len = int(segment_sec * SAMPLE_RATE)
    pause = np.zeros(int(pause_sec * SAMPLE_RATE), dtype=np.float32)
    first = fit_audio_length(first_speaker, segment_len)
    second = fit_audio_length(second_speaker, segment_len)
    return peak_normalize(np.concatenate([first, pause, second]).astype(np.float32))


dialogue_audio = build_safe_dialogue_audio(clean_audio, payload_audio, segment_sec=5.0, pause_sec=0.25)
dialogue_transcript = transcribe_waveform(torch.tensor(dialogue_audio, dtype=torch.float32, device=DEVICE).unsqueeze(0))
dialogue_reference = normalize_text(clean_transcript + " " + payload_transcript)

dialogue_result = pd.DataFrame([{
    "attack_type": "MSD_safe",
    "dialogue_reference": dialogue_reference,
    "dialogue_transcript": normalize_text(dialogue_transcript),
    "wer_vs_dialogue_reference": transcript_wer(dialogue_reference, dialogue_transcript),
}])

display(dialogue_result.style.format({
    "wer_vs_dialogue_reference": "{:.3f}",
}))
print("Safe two-speaker dialogue audio")
display(Audio(dialogue_audio, rate=SAMPLE_RATE))


## 7. Limitations

This notebook demonstrates empirical adversarial and compositional stress tests, not formal robustness certificates.

- The PGD perturbation is bounded by `L_inf`, but human imperceptibility is not formally proven.
- The composition section is inspired by SACRED-Bench, but it uses only benign content and evaluates ASR transcription changes rather than multimodal harmful compliance.
- Composition attacks are black-box with respect to the model, while the PGD section is white-box because it uses gradients.
- Results depend on the chosen sample, model, epsilon, composition parameters, and number of optimization steps.
- Audio players are included so a human can inspect whether the modified audio still sounds acceptable.
- This should be used for robustness research and defensive evaluation, not for deploying deceptive audio.
